In [175]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.PandasTools import LoadSDF
import tmap as tm
from map4 import MAP4Calculator
import warnings, json, traceback, csv
import pickle
from progress_bar import print_progress_bar
warnings.filterwarnings('ignore')

In [6]:
def pickle_dump(file_name, data):
    with open(file_name, 'wb') as f:
        pickle.dump(data, f)
def pickle_load(file_name):
    with open(file_name, 'rb') as f:
        return pickle.load(f)

In [3]:
DATA_FOLDER = 'database_unification_data/'

In [37]:
def fp_similarity_indiv(df_to_test, id_to_test_column_name, df_ids, id_column_name):
    '''
    Function that compares compound fingerprints and gets the distance between each of them.
    If the distance is 1, the compound database id is added to the ids dataframe. If there are no compounds with distance 1,
    the compound is appended to the ids dataframe.
    '''
    dim = 1024
    MAP4 = MAP4Calculator(dimensions=dim)
    ENC = tm.Minhash(dim)

    df_test = df_to_test.copy()
    df_total = df_ids.copy()

    print_progress_bar(0, len(df_test), suffix=f'(0/{len(df_test)})')
    for i in range(len(df_test)):
        try:
            current_fp = MAP4.calculate(df_test.iloc[i]['Molecule'])
        except Exception as e1:
            current_fp = None
            print(f'Error calculating fp for compound {i} in dataframe to test:\n{str(e1)}')
        found = False
        if current_fp:
            for j in range(len(df_total)):
                try:
                    compared_fp = MAP4.calculate(df_total.iloc[j]['Molecule'])
                except Exception as e2:
                    compared_fp = None
                    print(f'Error calculating fp for compound {j} in ids dataframe:\n{str(e2)}')
                if compared_fp:
                    current_distance = ENC.get_distance(current_fp, compared_fp)
                    if current_distance == 0:
                        df_ids.iloc[j][id_column_name] = df_to_test.iloc[i][id_to_test_column_name]
                        found = True
                        break
            if not found:
                new_df_row = {'ID': len(df_ids), id_column_name: df_test.iloc[i][id_to_test_column_name], 'Molecule': df_test.iloc[i]['Molecule'], 'SMILES': df_test.iloc[i]['SMILES']}
                df_ids = df_ids.append(new_df_row, ignore_index=True)
        print_progress_bar(i + 1, len(df_test), suffix=f'({i}/{len(df_test)})')
    
    return df_ids

In [156]:
# Load the BUL (Biological unit library) or Natural products library
with open('data/BUL_names.pkl', 'rb') as f:
    BULnames = pickle.load(f)

In [157]:
BUL_ids = list(set(list(sum(BULnames.keys(), ()))))

In [158]:
len(BUL_ids)

21057

In [153]:
BUL_ids

['C13750',
 'C20798',
 'C06007',
 'C06099',
 'C16207',
 'C10719',
 'C10586',
 'C07837',
 'C11169',
 'C00437',
 'C21594',
 '134812127_mod_0',
 'C02656_mod_2',
 'C22102',
 'C22141',
 'C01015',
 'C10708',
 'C15117',
 'C12115',
 'C00771',
 'C08885',
 'C10093',
 'C06033',
 'C04081',
 'C21923',
 'C12344',
 'C03843',
 'C15902',
 'C15996',
 'C17747',
 'C19828',
 'C18035',
 'C15579',
 'C10558',
 'C14937',
 'C13103',
 '162985355_mod_0',
 'C07006',
 'C11133',
 'C10350',
 'C20331',
 'C18642',
 'C04615',
 'C14731',
 'C02821',
 'C19409',
 'C00725',
 'C00025_mod_0',
 'C08759',
 '146682617_mod_0',
 '162962840_mod_0',
 'C16526',
 'C15137',
 '78383960',
 '146682760_mod_99',
 'C14577',
 'C17146',
 'C20248',
 'C13591',
 'C09326',
 '5282597',
 'C09333',
 'C01590',
 'C02457',
 'C08580',
 'C20815',
 'C17827',
 'C17497',
 '146683552_mod_82',
 'C07381',
 'C04644',
 'C14573',
 'C11881',
 '163030178_mod_0',
 '146682760_mod_97',
 'C20947',
 'C18988',
 '5312644_mod_0',
 'C01270',
 'C01279',
 'C18545',
 'C17722',
 

In [159]:
BUL_ids_filtered = [id for id in BUL_ids if id.startswith('C')]

In [169]:
BUL_cids = [id for id in BUL_ids if not id.startswith('C')]

In [170]:
BUL_cids

['134812127_mod_0',
 '162985355_mod_0',
 '146682617_mod_0',
 '162962840_mod_0',
 '78383960',
 '146682760_mod_99',
 '5282597',
 '146683552_mod_82',
 '163030178_mod_0',
 '146682760_mod_97',
 '5312644_mod_0',
 '138454203_mod_0',
 '135002_mod_0',
 '23250',
 '11463588',
 '146682758_mod_26',
 '73242171_mod_0',
 '146682758',
 '146683550_mod_13',
 '75201659',
 '14427323',
 '18676629_mod_3',
 '52921744',
 '146682621_mod_6',
 '52921860',
 '10982228',
 '53813767',
 '74611',
 '95300',
 '123364114',
 '24786700',
 '5282938_mod_0',
 '5282610_mod_0',
 '151922',
 '399_mod_5',
 '5312334_mod_0',
 '76863_mod_0',
 '162875369_mod_1',
 '146683552_mod_89',
 '5288377',
 '138454173_mod_0',
 '5312648_mod_0',
 '163064282_mod_0',
 '5312921_mod_0',
 '129726852_mod_0',
 '138454204',
 '2724554_mod_2',
 '18676629_mod_2',
 '5469810',
 '146682760_mod_36',
 '76518481_mod_0',
 '71402234_mod_0',
 '73721023_mod_0',
 '75009',
 '309_mod_7',
 '10424556',
 '5282787',
 '162900451_mod_2',
 '146683550_mod_0',
 '3632876',
 '1466835

In [160]:
BUL_ids_filtered

['C13750',
 'C20798',
 'C06007',
 'C06099',
 'C16207',
 'C10719',
 'C10586',
 'C07837',
 'C11169',
 'C00437',
 'C21594',
 'C02656_mod_2',
 'C22102',
 'C22141',
 'C01015',
 'C10708',
 'C15117',
 'C12115',
 'C00771',
 'C08885',
 'C10093',
 'C06033',
 'C04081',
 'C21923',
 'C12344',
 'C03843',
 'C15902',
 'C15996',
 'C17747',
 'C19828',
 'C18035',
 'C15579',
 'C10558',
 'C14937',
 'C13103',
 'C07006',
 'C11133',
 'C10350',
 'C20331',
 'C18642',
 'C04615',
 'C14731',
 'C02821',
 'C19409',
 'C00725',
 'C00025_mod_0',
 'C08759',
 'C16526',
 'C15137',
 'C14577',
 'C17146',
 'C20248',
 'C13591',
 'C09326',
 'C09333',
 'C01590',
 'C02457',
 'C08580',
 'C20815',
 'C17827',
 'C17497',
 'C07381',
 'C04644',
 'C14573',
 'C11881',
 'C20947',
 'C18988',
 'C01270',
 'C01279',
 'C18545',
 'C17722',
 'C21580',
 'C14634',
 'C19242',
 'C17107',
 'C09029',
 'C18500',
 'C00250',
 'C15373',
 'C14933',
 'C15168',
 'C11360',
 'C21107',
 'C19164',
 'C07034',
 'C21772',
 'C19482',
 'C01991',
 'C10628',
 'C17837'

In [199]:
len(BUL_ids_filtered)

17073

In [200]:
BUL_ids_filtered_no_mods = [id for id in BUL_ids_filtered if 'mod' not in id]

In [201]:
BUL_ids_filtered_no_mods

['C13750',
 'C20798',
 'C06007',
 'C06099',
 'C16207',
 'C10719',
 'C10586',
 'C07837',
 'C11169',
 'C00437',
 'C21594',
 'C22102',
 'C22141',
 'C01015',
 'C10708',
 'C15117',
 'C12115',
 'C00771',
 'C08885',
 'C10093',
 'C06033',
 'C04081',
 'C21923',
 'C12344',
 'C03843',
 'C15902',
 'C15996',
 'C17747',
 'C19828',
 'C18035',
 'C15579',
 'C10558',
 'C14937',
 'C13103',
 'C07006',
 'C11133',
 'C10350',
 'C20331',
 'C18642',
 'C04615',
 'C14731',
 'C02821',
 'C19409',
 'C00725',
 'C08759',
 'C16526',
 'C15137',
 'C14577',
 'C17146',
 'C20248',
 'C13591',
 'C09326',
 'C09333',
 'C01590',
 'C02457',
 'C08580',
 'C20815',
 'C17827',
 'C17497',
 'C07381',
 'C04644',
 'C14573',
 'C11881',
 'C20947',
 'C18988',
 'C01270',
 'C01279',
 'C18545',
 'C17722',
 'C21580',
 'C14634',
 'C19242',
 'C17107',
 'C09029',
 'C18500',
 'C00250',
 'C15373',
 'C14933',
 'C15168',
 'C11360',
 'C21107',
 'C19164',
 'C07034',
 'C21772',
 'C19482',
 'C01991',
 'C10628',
 'C17837',
 'C11015',
 'C17826',
 'C20365',

In [202]:
len(BUL_ids_filtered_no_mods)

16612

In [171]:
BUL_cids_no_mods = [id for id in BUL_cids if 'mod' not in id]

In [203]:
len(BUL_cids_no_mods)

1727

In [178]:
BUL_cids_no_mods

['78383960',
 '5282597',
 '23250',
 '11463588',
 '146682758',
 '75201659',
 '14427323',
 '52921744',
 '52921860',
 '10982228',
 '53813767',
 '74611',
 '95300',
 '123364114',
 '24786700',
 '151922',
 '5288377',
 '138454204',
 '5469810',
 '75009',
 '10424556',
 '5282787',
 '3632876',
 '5312578',
 '20848956',
 '549854',
 '162911315',
 '162867002',
 '162927483',
 '85699046',
 '182089',
 '12575964',
 '78360',
 '73856880',
 '129716173',
 '5282457',
 '163057816',
 '53262386',
 '5282650',
 '163088984',
 '56839943',
 '5312427',
 '5312781',
 '6439582',
 '138454199',
 '162985202',
 '53262345',
 '5282808',
 '5282595',
 '5282172',
 '163020806',
 '56633806',
 '162934221',
 '162979714',
 '5282773',
 '3035421',
 '195555',
 '5282596',
 '5282875',
 '5470168',
 '12593478',
 '44256492',
 '23427391',
 '5281932',
 '138454216',
 '52921746',
 '71338159',
 '14376443',
 '57494716',
 '6506600',
 '10924449',
 '5282750',
 '14211733',
 '92033282',
 '162970135',
 '10660115',
 '72992343',
 '91856968',
 '5312724',
 '5

In [180]:
with open('BUL_cids.csv', 'w') as f:
    f.write('CID\n')
    f.write('\n'.join(BUL_cids_no_mods))

In [185]:
df_metanetx_ref_concat = pd.read_csv('metanetx_ref_concat.csv')

In [186]:
df_metanetx_ref_concat

,ID,concat,description
0,BIOMASS,"seed.compound:cpd11416,seedM:M_cpd11416,seedM:...",Biomass
1,MNXM01,"MNXM01,mnx:PMF",PMF||Translocated proton that acccounts for th...
2,MNXM1,"CHEBI:15378,CHEBI:24636,CHEBI:29233,CHEBI:2923...",H+||Hydron||Proton||h||hydrogen ion||proton
3,MNXM10,"CHEBI:16908,CHEBI:57945,bigg.metabolite:nadh,b...",NADH||DPNH||NAD-reduced||NADH+H+||NADH2||Nicot...
4,MNXM100,"CHEBI:17211,CHEBI:58057,SLM:000000305,bigg.met...",geranyl diphosphate
...,...,...,...
42549,MNXM9995,"metacyc.compound:CPD-10654,metacycM:CPD-10654,...",1-chloro-3-methylbenzene||3-chlorotoluene
42550,MNXM99969,envipath:650babc9-9d68-4b73-9332-11972ca26f7b/...,n-pentatriacontane||pentatriacontane
42551,MNXM9999,"CHEBI:84431,chebi:84431,metacyc.compound:CPD-9...","2-heptaprenyl-3-methyl-5-hydroxy-6-methoxy-1,4..."
42552,MNXM99992,"lipidmaps:LMFA07010321,lipidmapsM:LMFA07010321...","(9Z,11E)-tetradeca-9,11-dien-1-yl acetate"


In [7]:
df_lotus = pickle_load(f'{DATA_FOLDER}pickles/lotus.pkl')

In [10]:
metanetx_tsv_path = f'{DATA_FOLDER}chem_prop.tsv'
df_metanetx = pd.read_csv(metanetx_tsv_path, sep='\t', header=None, skiprows=353, names=['ID', 'name', 'reference', 'formula', 'charge', 'mass', 'InChI', 'InChIKey', 'SMILES'])

In [11]:
df_metanetx = df_metanetx[df_metanetx['SMILES'].notna()]
df_metanetx = df_metanetx.reset_index(drop=True)

In [12]:
df_metanetx['Molecule'] = df_metanetx.apply(lambda x: Chem.MolFromSmiles(x['SMILES']), axis=1)

RDKit WARNING: [16:36:53] WARNING: not removing hydrogen atom without neighbors
[16:36:53] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [16:36:53] WARNING: not removing hydrogen atom without neighbors
[16:36:53] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [16:37:05] WARNING: not removing hydrogen atom without neighbors
[16:37:05] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [16:37:05] WARNING: not removing hydrogen atom without neighbors
[16:37:05] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [16:37:06] WARNING: not removing hydrogen atom without neighbors
[16:37:06] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [16:37:06] WARNING: not removing hydrogen atom without neighbors
[16:37:06] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [16:37:07] WARNING: not removing hydrogen atom without neighbors
[16:37:07] WARNING: not removing hydrogen atom wit

In [13]:
df_metanetx = df_metanetx[['ID', 'SMILES', 'Molecule']]

In [ ]:
# df_metanetx = pickle_load(f'{DATA_FOLDER}pickles/metanetx.pkl')

In [187]:
len(df_metanetx)

1248094

In [188]:
df_metanetx_filtered = df_metanetx[df_metanetx['ID'].isin(df_metanetx_ref_concat['ID'])]

In [189]:
df_metanetx_filtered

,ID,SMILES,Molecule
0,MNXM01,[H+],"<img data-content=""rdkit/molecule"" src=""data:i..."
3,MNXM1,[H+],"<img data-content=""rdkit/molecule"" src=""data:i..."
4,MNXM10,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)([O-])OP(=O)(...,"<img data-content=""rdkit/molecule"" src=""data:i..."
5,MNXM100,CC(C)=CCC/C(C)=C/COP(=O)([O-])OP(=O)([O-])[O-],"<img data-content=""rdkit/molecule"" src=""data:i..."
225,MNXM10002,C=C(C)[C@@H]1CC=C2[C@H](O)CC[C@@H](C)[C@@]2(C)C1,"<img data-content=""rdkit/molecule"" src=""data:i..."
...,...,...,...
1247541,MNXM9995,Cc1cccc(Cl)c1,"<img data-content=""rdkit/molecule"" src=""data:i..."
1247751,MNXM99969,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,"<img data-content=""rdkit/molecule"" src=""data:i..."
1247982,MNXM9999,COc1c(O)c(O)c(C)c(C/C=C(\C)CC/C=C(\C)CC/C=C(\C...,"<img data-content=""rdkit/molecule"" src=""data:i..."
1248005,MNXM99992,CC/C=C/C=C\CCCCCCCCOC(C)=O,"<img data-content=""rdkit/molecule"" src=""data:i..."


In [195]:
df_metanetx_filtered = df_metanetx_filtered.reset_index(drop=True)

In [196]:
pickle_dump(f'{DATA_FOLDER}pickles/metanetx_filtered.pkl', df_metanetx_filtered)

In [197]:
df_mtnx_aux = pickle_load(f'{DATA_FOLDER}pickles/metanetx_filtered.pkl')

In [198]:
df_mtnx_aux

,ID,SMILES,Molecule
0,MNXM01,[H+],"<img data-content=""rdkit/molecule"" src=""data:i..."
1,MNXM1,[H+],"<img data-content=""rdkit/molecule"" src=""data:i..."
2,MNXM10,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)([O-])OP(=O)(...,"<img data-content=""rdkit/molecule"" src=""data:i..."
3,MNXM100,CC(C)=CCC/C(C)=C/COP(=O)([O-])OP(=O)([O-])[O-],"<img data-content=""rdkit/molecule"" src=""data:i..."
4,MNXM10002,C=C(C)[C@@H]1CC=C2[C@H](O)CC[C@@H](C)[C@@]2(C)C1,"<img data-content=""rdkit/molecule"" src=""data:i..."
...,...,...,...
27644,MNXM9995,Cc1cccc(Cl)c1,"<img data-content=""rdkit/molecule"" src=""data:i..."
27645,MNXM99969,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,"<img data-content=""rdkit/molecule"" src=""data:i..."
27646,MNXM9999,COc1c(O)c(O)c(C)c(C/C=C(\C)CC/C=C(\C)CC/C=C(\C...,"<img data-content=""rdkit/molecule"" src=""data:i..."
27647,MNXM99992,CC/C=C/C=C\CCCCCCCCOC(C)=O,"<img data-content=""rdkit/molecule"" src=""data:i..."


In [73]:
with open('metanetx simplified.txt', 'r') as txt_file:
    mnx_ids = txt_file.read()
mnx_ids = mnx_ids.split("\n")
mnx_ids = [mnx_id.replace('@MNXD1', '').replace('@MNXD2', '') for mnx_id in mnx_ids]
mnx_ids = list(set([mnx_id for mnx_id in mnx_ids if mnx_id])) # Get rid of duplicates and empty spaces

In [71]:
df_metanetx_ref = pd.read_csv(f'{DATA_FOLDER}chem_xref.tsv', sep='\t', header=None, skiprows=354, names=['source', 'ID', 'description'])

In [83]:
df_metanetx_ref = df_metanetx_ref[df_metanetx_ref['ID'].isin(mnx_ids)]

In [88]:
ref_ids = df_metanetx_ref['ID'].tolist()
mnx_ids_not_included = [mnx_id for mnx_id in mnx_ids if mnx_id not in ref_ids]

In [89]:
mnx_ids_not_included

[]

In [84]:
len(df_metanetx_ref)

272034

In [115]:
len(mnx_ids)

42554

In [114]:
len(df_metanetx_ref['ID'].unique())

42554

In [126]:
len(df_metanetx_ref)

272034

In [142]:
df_metanetx_bul_filtered = df_metanetx_ref[df_metanetx_ref['source'].str.contains('|'.join(BUL_ids))]

In [137]:
df_metanetx_bul_filtered = df_metanetx_bul_filtered[~df_metanetx_bul_filtered['ID'].str.lower().duplicated()]

In [141]:
len(df_metanetx_bul_filtered)

245871

In [138]:
df_metanetx_bul_filtered

,source,ID,description
53,kegg.compound:C00080,MNXM1,H+||Hydron
128,kegg.compound:C00004,MNXM10,NADH||DPNH||Reduced nicotinamide adenine dinuc...
166,kegg.compound:C00341,MNXM100,Geranyl diphosphate
1552,kegg.compound:C16496,MNXM10006,3-Epiecdysone
2467,kegg.compound:C05834,MNXM10010,3-Methyldioxyindole||3-Hydroxy-3-methyloxindole
...,...,...,...
2992473,kegg.compound:C20838,MNXM9983,"3-[(3aS,4S,7aS)-7a-Methyl-1,5-dioxo-octahydro-..."
2993167,kegg.compound:C06057,MNXM9986,"3-Aminopropane-1,2-diol||(R)-3-Amino-1,2-propa..."
2993414,kegg.compound:C05145,MNXM9987,3-Aminoisobutyric acid||3-Amino-2-methylpropan...
2994130,kegg.compound:C16266,MNXM9990,3-Chloro-2-hydroxymuconic semialdehyde


In [149]:
df_metanetx_bul_filtered_final = df_metanetx_ref[~df_metanetx_ref['ID'].isin(df_metanetx_bul_filtered['ID'])]

In [147]:
df_metanetx_bul_filtered_final = [~df_metanetx_bul_filtered_final.isin(df_metanetx_bul_filtered_final['ID'] == 'BIOMASS')]

In [150]:
df_metanetx_bul_filtered_final

,source,ID,description
0,seed.compound:cpd11416,BIOMASS,Biomass
1,seedM:M_cpd11416,BIOMASS,secondary/obsolete/fantasy identifier
2,seedM:cpd11416,BIOMASS,Biomass
3,MNXM01,MNXM01,PMF||Translocated proton that acccounts for th...
4,mnx:PMF,MNXM01,PMF||Translocated proton that acccounts for th...
...,...,...,...
2996240,metacyc.compound:CPD-17804,MNXM99992,"(9Z,11E)-tetradeca-9,11-dien-1-yl acetate"
2996241,metacycM:CPD-17804,MNXM99992,"(9Z,11E)-tetradeca-9,11-dien-1-yl acetate"
2996242,seed.compound:cpd33563,MNXM99992,"(9Z,11E)-tetradeca-9,11-dien-1-yl acetate"
2996243,seedM:M_cpd33563,MNXM99992,secondary/obsolete/fantasy identifier


In [122]:
df_metanetx_ref.to_csv('metanetx_ref.csv')

In [102]:
df_metanetx_ref_final = df_metanetx_ref[~df_metanetx_ref['source'].str.lower().str.contains('lipidmaps')]

In [104]:
df_metanetx_ref_final = df_metanetx_ref_final[~df_metanetx_ref_final['source'].str.lower().str.contains('chebi:')]

In [105]:
df_metanetx_ref_final = df_metanetx_ref_final[~df_metanetx_ref_final['source'].str.lower().str.contains('kegg')]

In [107]:
df_metanetx_ref_final = df_metanetx_ref_final[~df_metanetx_ref_final['source'].str.lower().duplicated()]

In [108]:
len(df_metanetx_ref_final)

170634

In [109]:
len(df_metanetx_ref_final[df_metanetx_ref_final['source'].str.lower().str.contains('slm')])

1902

In [110]:
df_metanetx_ref_final.head()

,source,ID,description
0,seed.compound:cpd11416,BIOMASS,Biomass
1,seedM:M_cpd11416,BIOMASS,secondary/obsolete/fantasy identifier
2,seedM:cpd11416,BIOMASS,Biomass
3,MNXM01,MNXM01,PMF||Translocated proton that acccounts for th...
4,mnx:PMF,MNXM01,PMF||Translocated proton that acccounts for th...


In [15]:
len(df_lotus)

276518

In [14]:
len(df_metanetx)

1248094

In [16]:
# Populate compounds ids list with first database (LOTUS)
df_ids = df_lotus.copy()
df_ids.rename({'lotus_id': 'LOTUS_ID'}, axis=1, inplace=True)
df_ids['ID'] = df_ids.index
df_ids['METANETX_ID'] = None
df_ids['PEPTIDE_ID'] = None
df_ids['RHEA_ID'] = None
df_ids['KEGG_ID'] = None
df_ids['LIPIDMAPS_ID'] = None
df_ids['PUBCHEM_ID'] = None

In [17]:
df_ids.head()

,LOTUS_ID,SMILES,Molecule,ID,METANETX_ID,PEPTIDE_ID,RHEA_ID,KEGG_ID,LIPIDMAPS_ID,PUBCHEM_ID
0,LTS0000001,CC(C)CCCCCC=CC=CC(O)=NC(C(O)=NC1CC(O)CCN=C(O)C...,"<img data-content=""rdkit/molecule"" src=""data:i...",0,None,None,None,None,None,None
1,LTS0000002,CC(C)=CC(O)=NCCc1ccc2c(c1)C=CC(C)(C)O2,"<img data-content=""rdkit/molecule"" src=""data:i...",1,None,None,None,None,None,None
2,LTS0000003,CC1(C)CCC2(C(=O)OC3OC(CO)C(O)C(O)C3O)CCC3(C)C(...,"<img data-content=""rdkit/molecule"" src=""data:i...",2,None,None,None,None,None,None
3,LTS0000004,CC(=O)OCC(CC(C)C)N=C(O)C(N=C(O)C1CCCN1C(=O)C(C...,"<img data-content=""rdkit/molecule"" src=""data:i...",3,None,None,None,None,None,None
4,LTS0000005,CC1=C(O)c2c(oc(-c3ccccc3)cc2=O)C(C)(C)C1=O,"<img data-content=""rdkit/molecule"" src=""data:i...",4,None,None,None,None,None,None


In [29]:
df_test = df_ids.iloc[:1000]

In [30]:
len(df_test)

1000

In [31]:
df_test.head()

,LOTUS_ID,SMILES,Molecule,ID,METANETX_ID,PEPTIDE_ID,RHEA_ID,KEGG_ID,LIPIDMAPS_ID,PUBCHEM_ID
0,LTS0000001,CC(C)CCCCCC=CC=CC(O)=NC(C(O)=NC1CC(O)CCN=C(O)C...,"<img data-content=""rdkit/molecule"" src=""data:i...",0,None,None,None,None,None,None
1,LTS0000002,CC(C)=CC(O)=NCCc1ccc2c(c1)C=CC(C)(C)O2,"<img data-content=""rdkit/molecule"" src=""data:i...",1,None,None,None,None,None,None
2,LTS0000003,CC1(C)CCC2(C(=O)OC3OC(CO)C(O)C(O)C3O)CCC3(C)C(...,"<img data-content=""rdkit/molecule"" src=""data:i...",2,None,None,None,None,None,None
3,LTS0000004,CC(=O)OCC(CC(C)C)N=C(O)C(N=C(O)C1CCCN1C(=O)C(C...,"<img data-content=""rdkit/molecule"" src=""data:i...",3,None,None,None,None,None,None
4,LTS0000005,CC1=C(O)c2c(oc(-c3ccccc3)cc2=O)C(C)(C)C1=O,"<img data-content=""rdkit/molecule"" src=""data:i...",4,None,None,None,None,None,None


In [32]:
df_metanetx_test = df_metanetx.iloc[:1000]

In [33]:
len(df_metanetx_test)

1000

In [35]:
df_metanetx_test.head()

,ID,SMILES,Molecule
0,MNXM01,[H+],"<img data-content=""rdkit/molecule"" src=""data:i..."
1,MNXM02,[O-][H],"<img data-content=""rdkit/molecule"" src=""data:i..."
2,MNXM03,[OH3+],"<img data-content=""rdkit/molecule"" src=""data:i..."
3,MNXM1,[H+],"<img data-content=""rdkit/molecule"" src=""data:i..."
4,MNXM10,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)([O-])OP(=O)(...,"<img data-content=""rdkit/molecule"" src=""data:i..."


In [70]:
# vs METANETX
# fp_similarity(fps_total, fps_metanetx, df_metanetx, 'ID', df_ids, 'METANETX_ID')
df_res = fp_similarity_indiv(df_metanetx.iloc[:1], 'ID', df_ids.iloc[:1000], 'METANETX_ID')
df_res

 |████████████████████████████████████████████████████████████████████████████████████████████████████| 100.0% (0/1)


,LOTUS_ID,SMILES,Molecule,ID,METANETX_ID,PEPTIDE_ID,RHEA_ID,KEGG_ID,LIPIDMAPS_ID,PUBCHEM_ID
0,LTS0000001,CC(C)CCCCCC=CC=CC(O)=NC(C(O)=NC1CC(O)CCN=C(O)C...,"<img data-content=""rdkit/molecule"" src=""data:i...",0,None,None,None,None,None,None
1,LTS0000002,CC(C)=CC(O)=NCCc1ccc2c(c1)C=CC(C)(C)O2,"<img data-content=""rdkit/molecule"" src=""data:i...",1,None,None,None,None,None,None
2,LTS0000003,CC1(C)CCC2(C(=O)OC3OC(CO)C(O)C(O)C3O)CCC3(C)C(...,"<img data-content=""rdkit/molecule"" src=""data:i...",2,None,None,None,None,None,None
3,LTS0000004,CC(=O)OCC(CC(C)C)N=C(O)C(N=C(O)C1CCCN1C(=O)C(C...,"<img data-content=""rdkit/molecule"" src=""data:i...",3,None,None,None,None,None,None
4,LTS0000005,CC1=C(O)c2c(oc(-c3ccccc3)cc2=O)C(C)(C)C1=O,"<img data-content=""rdkit/molecule"" src=""data:i...",4,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...
996,LTS0000997,COC(=O)C12Oc3cccc(O)c3C(=O)C1=CCC(C)C2O,"<img data-content=""rdkit/molecule"" src=""data:i...",996,None,None,None,None,None,None
997,LTS0000998,CC(=O)OC1CCC2(C)C3=C(CCC2C1(C)C)C1(C)CC(=O)C(C...,"<img data-content=""rdkit/molecule"" src=""data:i...",997,None,None,None,None,None,None
998,LTS0000999,Oc1ccc(C=Cc2cc(O)cc3c2C(c2cc(O)cc(O)c2)C(c2ccc...,"<img data-content=""rdkit/molecule"" src=""data:i...",998,None,None,None,None,None,None
999,LTS0001000,Cc1c(O)cc(C(=O)C(O)C2CCC(C)O2)cc1O,"<img data-content=""rdkit/molecule"" src=""data:i...",999,None,None,None,None,None,None


In [61]:
# vs LOTUS
df_res = fp_similarity_indiv(df_ids.iloc[499:500], 'LOTUS_ID', df_ids.iloc[:1000], 'LOTUS_ID')
df_res

 |████████████████████████████████████████████████████████████████████████████████████████████████████| 100.0% (0/1)


,LOTUS_ID,SMILES,Molecule,ID,METANETX_ID,PEPTIDE_ID,RHEA_ID,KEGG_ID,LIPIDMAPS_ID,PUBCHEM_ID
0,LTS0000001,CC(C)CCCCCC=CC=CC(O)=NC(C(O)=NC1CC(O)CCN=C(O)C...,"<img data-content=""rdkit/molecule"" src=""data:i...",0,None,None,None,None,None,None
1,LTS0000002,CC(C)=CC(O)=NCCc1ccc2c(c1)C=CC(C)(C)O2,"<img data-content=""rdkit/molecule"" src=""data:i...",1,None,None,None,None,None,None
2,LTS0000003,CC1(C)CCC2(C(=O)OC3OC(CO)C(O)C(O)C3O)CCC3(C)C(...,"<img data-content=""rdkit/molecule"" src=""data:i...",2,None,None,None,None,None,None
3,LTS0000004,CC(=O)OCC(CC(C)C)N=C(O)C(N=C(O)C1CCCN1C(=O)C(C...,"<img data-content=""rdkit/molecule"" src=""data:i...",3,None,None,None,None,None,None
4,LTS0000005,CC1=C(O)c2c(oc(-c3ccccc3)cc2=O)C(C)(C)C1=O,"<img data-content=""rdkit/molecule"" src=""data:i...",4,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...
995,LTS0000996,CC(=O)OC1CCC2(C)C(COc3ccc4ccc(=O)oc4c3)C(C)=CC...,"<img data-content=""rdkit/molecule"" src=""data:i...",995,None,None,None,None,None,None
996,LTS0000997,COC(=O)C12Oc3cccc(O)c3C(=O)C1=CCC(C)C2O,"<img data-content=""rdkit/molecule"" src=""data:i...",996,None,None,None,None,None,None
997,LTS0000998,CC(=O)OC1CCC2(C)C3=C(CCC2C1(C)C)C1(C)CC(=O)C(C...,"<img data-content=""rdkit/molecule"" src=""data:i...",997,None,None,None,None,None,None
998,LTS0000999,Oc1ccc(C=Cc2cc(O)cc3c2C(c2cc(O)cc(O)c2)C(c2ccc...,"<img data-content=""rdkit/molecule"" src=""data:i...",998,None,None,None,None,None,None


In [57]:
df_ids.iloc[1000:1001]

,LOTUS_ID,SMILES,Molecule,ID,METANETX_ID,PEPTIDE_ID,RHEA_ID,KEGG_ID,LIPIDMAPS_ID,PUBCHEM_ID
1000,LTS0001001,CC1CC(O)C2(O)OC3CC4(C)C(CCC5C4CCC4(C)C(C6=CC(=...,"<img data-content=""rdkit/molecule"" src=""data:i...",1000,None,None,None,None,None,None


## Check what KEGG ids are missing in MetanetX dataframe

In [204]:
# Dataframe with MetanetX molecules
df_mtnx_filtered = pickle_load(f'{DATA_FOLDER}pickles/metanetx_filtered.pkl')

In [206]:
# Dataframe with the databases ids linked to each MetanetX id
df_metanetx_ref_concat = pd.read_csv('metanetx_ref_concat.csv')

In [205]:
# Load the BUL (Biological unit library) or Natural products library
with open('data/BUL_names.pkl', 'rb') as f:
    BULnames = pickle.load(f)

BUL_ids = list(set(list(sum(BULnames.keys(), ())))) # Flattened list of current BUL ids
BUL_ids_filtered = [id for id in BUL_ids if id.startswith('C')] # List with only Kegg ids
BUL_ids_filtered_no_mods = [id for id in BUL_ids_filtered if 'mod' not in id] # List with Kegg ids with no mods

In [210]:
# Populate compounds ids list with first database (MetanetX)
df_ids = df_mtnx_filtered.copy()
df_ids.rename({'ID': 'METANETX_ID'}, axis=1, inplace=True)
df_ids['ID'] = df_ids.index
df_ids['LOTUS_ID'] = None
df_ids['PEPTIDE_ID'] = None
df_ids['RHEA_ID'] = None
df_ids['KEGG_ID'] = None
df_ids['LIPIDMAPS_ID'] = None
df_ids['PUBCHEM_ID'] = None

In [221]:
print_progress_bar(0, len(BUL_ids_filtered_no_mods), suffix=f'(0/{len(BUL_ids_filtered_no_mods)})')
for i, BUL_id in enumerate(BUL_ids_filtered_no_mods):
    df_aux = df_metanetx_ref_concat[df_metanetx_ref_concat['concat'].str.contains(BUL_id)]
    if df_aux.empty:
        kegg_mol = next((v for k, v in BULnames.items() if BUL_id in k), None)
        current_kegg_data = {'ID': len(df_ids), 'KEGG_ID': BUL_id, 'Molecule': kegg_mol, 'SMILES': Chem.MolToSmiles(kegg_mol)}
        df_ids = df_ids.append(current_kegg_data, ignore_index=True)
        # print(f'{BUL_id} not found! Inserted.')
    print_progress_bar(i + 1, len(BUL_ids_filtered_no_mods), suffix=f'({i + 1}/{len(BUL_ids_filtered_no_mods)})')

 |████████████████████████████████████████████████████████████████████████████████████████████████████| 100.0% (16612/16612)


In [222]:
df_ids

,METANETX_ID,SMILES,Molecule,ID,LOTUS_ID,PEPTIDE_ID,RHEA_ID,KEGG_ID,LIPIDMAPS_ID,PUBCHEM_ID
0,MNXM01,[H+],"<img data-content=""rdkit/molecule"" src=""data:i...",0,None,None,None,None,None,None
1,MNXM1,[H+],"<img data-content=""rdkit/molecule"" src=""data:i...",1,None,None,None,None,None,None
2,MNXM10,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)([O-])OP(=O)(...,"<img data-content=""rdkit/molecule"" src=""data:i...",2,None,None,None,None,None,None
3,MNXM100,CC(C)=CCC/C(C)=C/COP(=O)([O-])OP(=O)([O-])[O-],"<img data-content=""rdkit/molecule"" src=""data:i...",3,None,None,None,None,None,None
4,MNXM10002,C=C(C)[C@@H]1CC=C2[C@H](O)CC[C@@H](C)[C@@]2(C)C1,"<img data-content=""rdkit/molecule"" src=""data:i...",4,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...
35669,NaN,C[N+]1([O-])CC[C@]23c4c5ccc(O)c4O[C@H]2[C@@H](...,"<img data-content=""rdkit/molecule"" src=""data:i...",35669,NaN,NaN,NaN,C11786,NaN,NaN
35670,NaN,COc1cc(/C=C/C(=O)OC[C@H]2O[C@@H](Oc3cc4c(cc3O)...,"<img data-content=""rdkit/molecule"" src=""data:i...",35670,NaN,NaN,NaN,C08547,NaN,NaN
35671,NaN,CN1CCN(CCCN2c3ccccc3Sc3ccc(Cl)cc32)CC1,"<img data-content=""rdkit/molecule"" src=""data:i...",35671,NaN,NaN,NaN,C07403,NaN,NaN
35672,NaN,COc1cc(CC2c3c(cc(OC)c(OC)c3OC)CC[N+]2(C)CCCOC(...,"<img data-content=""rdkit/molecule"" src=""data:i...",35672,NaN,NaN,NaN,C07549,NaN,NaN


In [224]:
pickle_dump(f'{DATA_FOLDER}pickles/metanetx+kegg.pkl', df_ids)